# 3장 2강: Chain Rule과 Backpropagation — 실습문제

## 실습 목표

- 연결된 함수에서 중간값을 먼저 계산합니다.
- 주어진 미분 규칙으로 각 단계의 변화율을 구하고 곱합니다.
- 순전파의 계산값과 역전파의 기울기를 구별합니다.
- 교안의 작은 회귀·분류 모델에서 가중치와 편향의 기울기를 구합니다.
- 역전파와 파라미터를 실제로 바꾸는 과정이 다름을 설명합니다.

## 실습 환경 / 데이터

Python과 NumPy를 사용하며 외부 데이터 파일은 필요하지 않습니다. 제공된 「3장 2강: Chain Rule과 Backpropagation」 본문과 연습문제의 식·숫자를 사용합니다. 실제 구매는 y=1, 미구매는 y=0으로 표시합니다.

## 실습 구성과 교안 대응

| 구분 | 실습 내용 | 교안 근거 |
|---|---|---|
| 필수 1 / 문제 1-1 | f=(3x+1)², x=2의 두 단계 Chain Rule | 1절 본문: 합성함수와 특정 지점의 기울기 |
| 필수 1 / 문제 1-2 | 두 단계 회귀 모델의 순전파·역전파 | 2절 본문: x=2, y=12, w₁=2, b₁=1, w₂=2, b₂=0 |
| 필수 2 / 문제 2-1 | 실제 구매 고객의 Sigmoid+BCE 역전파 | 3절 본문: x=2, y=1, w=0.5, b=0 |
| 필수 2 / 문제 2-2 | 제공된 순서표로 ReLU 은닉층까지 연결 | 3절 본문: x=2, y=1, w₁=1, b₁=0, w₂=0.5, b₂=0 |
| 과제 / 문제 3-1 | f=(2x+3)², x=1의 기본 미분 복습 | 연습문제 1의 문제 1·2 |
| 과제 / 문제 3-2 | 실제 미구매 고객의 기본 Gradient 복습 | 연습문제 3의 문제 1~3 중 기본 계산 |

**교안의 연습문제 전체를 그대로 옮긴 자료는 아닙니다.** 본문 예제와 연습문제의 일부를 기초 실습 4문제와 복습 과제 2문제로 재구성했습니다. 새로운 미분법이나 교안에 없는 학습 알고리즘은 사용하지 않습니다.

## 진행 안내

1. 실습 준비 셀을 실행해 NumPy를 불러옵니다.
2. 문제에 제시된 식을 읽고 입력값부터 순서대로 계산합니다.
3. 제공된 미분식과 계산 순서표에 숫자를 대입합니다. 공식을 외우거나 혼자 유도할 필요는 없습니다.
4. 기울기를 계산할 때에는 **전달받은 손실 기울기 × 현재 연산의 미분값**을 구합니다.
5. 문제 2-2는 연결 단계가 많으므로 제공 표를 한 줄씩 따라가세요. 모든 단계를 한 번에 외울 필요는 없습니다.
6. 소수는 계산 중간에 반올림하지 않고 출력할 때만 소수점 아래 세 자리로 반올림합니다.
7. Q1~Q4에 답하세요. 숫자뿐 아니라 무엇을 계산한 값인지 함께 적습니다.

## 먼저 읽는 용어와 기호

| 용어·기호 | 쉬운 뜻 |
|---|---|
| 함수 / 함수값 | 입력으로 출력을 계산하는 규칙 / 그 규칙에 숫자를 넣은 결과 |
| 합성함수 | 한 함수의 출력이 다음 함수의 입력으로 들어가는 구조 |
| 중간 변수 z, a, h | 연산 사이에서 결과를 보관하는 이름. 정확한 뜻은 각 문제의 식에서 확인 |
| 미분값·기울기 | 현재 점에서 입력을 아주 조금 바꿀 때 출력이 변하는 방향과 정도 |
| 국소 미분값 | 전체 계산 중 한 연산만 따로 본 변화율. ‘국소’는 여기서 ‘한 단계’라는 뜻 |
| Chain Rule / 연쇄법칙 | 연결된 단계의 미분값을 곱해 전체 변화율을 구하는 규칙 |
| 계산 그래프 | 어떤 값이 어떤 연산을 거쳐 다음 값에 연결되는지 나타낸 구조 |
| 순전파 | 입력부터 중간값·예측·손실을 순서대로 계산하는 과정 |
| 역전파 / Backpropagation | 최종 손실부터 이전 연산으로 기울기를 전달하는 계산 과정 |
| x / y | 입력 데이터 / 실제 정답. 문제에서 주어진 고정값 |
| w / b | 입력에 곱하는 가중치 / 더하는 편향. 모델에서 조절하는 수인 파라미터 |
| w₁, b₁ / w₂, b₂ | 첫 번째 / 두 번째 연산의 파라미터. 아래 숫자는 단계 표시이며 제곱이 아님 |
| ŷ / y_pred | 모델의 회귀 예측값. ŷ는 ‘와이 햇’으로 읽음 |
| L / loss | 예측을 정답과 비교해 계산한 손실 |
| Logit z | Sigmoid로 확률을 만들기 전 점수. z 자체를 확률로 읽지 않음 |
| Sigmoid / p | 점수를 0과 1 사이 확률로 바꾸는 함수 / 이 자료에서는 구매 확률 |
| BCE | 구매·미구매처럼 두 종류를 예측할 때 사용하는 손실 |
| ReLU | 입력이 양수이면 그대로, 음수이면 0을 내보내는 함수 |
| 은닉층 / h | 입력과 최종 출력 사이의 중간 처리 단계 / 이 자료에서는 ReLU를 거친 중간값 |
| Gradient / ∇L | 파라미터별 손실 기울기를 정해진 순서대로 묶은 것 |
| 파라미터 업데이트 | 계산한 기울기를 이용해 w, b 등의 값을 실제로 변경하는 일 |
| ≈ / ln | 대략 같음 / 자연로그 |

### 미분 기호를 읽는 방법

`df/dz`는 **z에 대한 f의 순간 변화율**입니다. 분수처럼 보여도 f의 값에서 z의 값을 단순히 나누라는 뜻은 아닙니다. `∂L/∂w`는 다른 독립적인 변수는 고정하고 w를 바꿀 때의 손실 변화율입니다. w의 변화가 중간값을 거쳐 L까지 전달되는 영향은 포함합니다.

코드의 `dL_dw`는 `∂L/∂w`를 저장하는 이름입니다. `dL_dz`는 z에 대한 손실 기울기이고 `dp_dz`는 z에 대한 확률의 변화율입니다. **앞의 문자와 뒤의 문자를 모두 읽어야 무엇의 변화율인지 알 수 있습니다.**

### 사용할 기본 미분 규칙

| 연산 | 미분 대상 | 미분한 결과 |
|---|---|---|
| z=3x+1 | x | 3 |
| f=z² | z | 2z |
| ŷ=wx+b | w / b | x / 1 |
| L=(y−ŷ)² | ŷ | 2(ŷ−y), 실제 정답 y는 고정 |
| p=Sigmoid(z) | z | p(1−p) |
| Sigmoid와 BCE를 연결한 L | z | p−y |

함수값을 계산하는 식과 미분값을 계산하는 식을 구분하세요. 예를 들어 f=z²의 출력은 z²이고, 그 연산의 미분값은 2z입니다.

**지수·로그가 처음인 경우:** e는 약 2.718인 수이고 `np.exp(t)`는 e의 t제곱을 계산합니다. 자연로그 ln은 e를 몇 제곱하면 주어진 수가 되는지 나타내는 계산입니다. 이번 실습에서는 지수·로그 값을 손으로 구하지 않고 제공된 코드를 사용합니다.

## 실습 준비

아래 셀을 먼저 실행하세요. NumPy는 숫자 계산을 돕는 도구이고, `np`는 NumPy를 짧게 부르는 이름입니다.

In [1]:
import numpy as np

---

## 필수 실습 1. Chain Rule과 역전파의 기본

### 문제 1-1. 두 단계 함수의 변화율 연결하기

#### 문제 설명

교안의 합성함수 $f(x)=(3x+1)^2$에서 x=2를 사용합니다. 계산을 두 단계로 나눕니다.

$$z=3x+1,\qquad f=z^2$$

먼저 z를 계산하고 그 결과를 제곱합니다. 미분할 때는 다음 식을 사용합니다.

$$\frac{df}{dz}=2z,\qquad \frac{dz}{dx}=3,\qquad
\frac{df}{dx}=\frac{df}{dz}\times\frac{dz}{dx}$$

#### 요구사항

1. x=2에서 z와 f를 차례로 구하세요.
2. df/dz와 dz/dx를 각각 구하고 곱해 df/dx를 계산하세요.
3. 계산 결과를 출력하고 Q1~Q4에 답하세요.

#### 작성 도움말

```python
x = 2.0
z = 3 * x + 1
f = z ** 2
df_dz = 2 * z
dz_dx = 3.0
df_dx = df_dz * dz_dx
```

이 코드를 참고해 결과를 출력하세요. `df_dz`와 `df_dx`는 서로 다른 변화율입니다. 일반적인 도함수를 혼자 유도하는 것은 요구하지 않습니다.

#### 해석 질문

**Q1.** 중간값 z와 함수값 f는 각각 얼마인가요?

**Q2.** 두 국소 미분값 df/dz, dz/dx와 전체 미분값 df/dx는 얼마인가요?

**Q3.** df/dx를 구할 때 14만 쓰면 어떤 계산을 빠뜨리나요?

**Q4.** 함수값 49와 미분값 42는 어떤 차이가 있나요?

#### 제출 결과

- 요구사항의 계산 결과
- Q1~Q4 답변

In [2]:
# 문제 1-1. 안쪽 함수와 바깥 함수의 미분값을 곱합니다.
x = 2.0
z = 3 * x + 1
f = z ** 2
df_dz = 2 * z
dz_dx = 3.0
df_dx = df_dz * dz_dx

print("중간값 z:", z)
print("함수값 f:", f)
print("df/dz:", df_dz)
print("dz/dx:", dz_dx)
print("df/dx:", df_dx)

중간값 z: 7.0
함수값 f: 49.0
df/dz: 14.0
dz/dx: 3.0
df/dx: 42.0


#### 풀이 및 답변

**수식 및 계산 과정**

- 순전파: $z=3\times2+1=7$, $f=z^2=7^2=49$입니다.
- 바깥 함수 미분은 $df/dz=2z=14$, 안쪽 함수 미분은 $dz/dx=3$입니다.
- Chain Rule로 $df/dx=14\times3=42$입니다.

**Q1.** 중간값 z와 함수값 f는 각각 얼마인가요?

**답변:** 중간값 z는 7이고 함수값 f는 49입니다.

**Q2.** 두 국소 미분값 df/dz, dz/dx와 전체 미분값 df/dx는 얼마인가요?

**답변:** $df/dz=14$, $dz/dx=3$이며, 전체 미분값은 $df/dx=14\times3=42$입니다.

**Q3.** df/dx를 구할 때 14만 쓰면 어떤 계산을 빠뜨리나요?

**답변:** 안쪽 함수 $z=3x+1$에서 x가 z에 미치는 변화율 $dz/dx=3$을 곱하는 단계를 빠뜨립니다. 14는 z에 대한 f의 기울기입니다.

**Q4.** 함수값 49와 미분값 42는 어떤 차이가 있나요?

**답변:** 49는 x=2를 넣었을 때의 함수 출력이고, 42는 그 점에서 x를 아주 조금 바꿀 때 f가 변하는 순간 변화율입니다.

### 문제 1-2. 두 단계 회귀 모델의 역전파 따라가기

#### 문제 설명

교안의 두 단계 회귀 모델입니다. 회귀는 숫자로 된 결과를 예측하는 문제입니다.

$$a=w_1x+b_1,\qquad \hat y=w_2a+b_2,\qquad L=(y-\hat y)^2$$

| x | y | w₁ | b₁ | w₂ | b₂ |
|---:|---:|---:|---:|---:|---:|
| 2 | 12 | 2 | 1 | 2 | 0 |

#### 요구사항

1. a, 예측값 ŷ, 손실 L을 순서대로 계산하세요.
2. 아래 표의 위에서 아래로 기울기를 계산하세요. 오른쪽 식이 계산 도움말입니다.
3. Gradient를 [w₁,b₁,w₂,b₂] 순서로 묶어 출력하고 Q1~Q4에 답하세요.

| 순서 | 계산 대상 | 사용할 식 |
|---|---|---|
| 1 | 예측값에 대한 손실 기울기 | `dL_dy_pred = 2 * (y_pred - y)` |
| 2 | 두 번째 가중치·편향 | `dL_dw2 = dL_dy_pred * a`, `dL_db2 = dL_dy_pred * 1` |
| 3 | 앞 연산의 출력 a | `dL_da = dL_dy_pred * w2` |
| 4 | 첫 번째 가중치·편향 | `dL_dw1 = dL_da * x`, `dL_db1 = dL_da * 1` |

#### 작성 도움말

순전파는 `a = w1 * x + b1`, `y_pred = w2 * a + b2`, `loss = (y - y_pred) ** 2`입니다. 표의 식을 그대로 참고해도 됩니다. w₂의 기울기에는 a를 곱하고, a로 전달할 기울기에는 w₂를 곱합니다. 변수의 값을 변경하는 업데이트는 요구하지 않습니다.

#### 해석 질문

**Q1.** 순전파의 중간값 a, 예측값 ŷ, 손실 L은 각각 얼마인가요?

**Q2.** ∂L/∂ŷ와 두 번째 연산의 ∂L/∂w₂, ∂L/∂b₂는 얼마인가요?

**Q3.** a에 전달한 기울기와 첫 번째 연산의 두 기울기는 얼마인가요?

**Q4.** 역전파를 계산하면 가중치와 편향이 자동으로 바뀌나요?

#### 제출 결과

- 요구사항의 계산 결과
- Q1~Q4 답변

In [3]:
# 문제 1-2. 순전파의 값을 저장하고 손실에서부터 거꾸로 계산합니다.
x, y = 2.0, 12.0
w1, b1, w2, b2 = 2.0, 1.0, 2.0, 0.0
a = w1 * x + b1
y_pred = w2 * a + b2
loss = (y - y_pred) ** 2

dL_dy_pred = 2 * (y_pred - y)
dL_dw2 = dL_dy_pred * a
dL_db2 = dL_dy_pred * 1
dL_da = dL_dy_pred * w2
dL_dw1 = dL_da * x
dL_db1 = dL_da * 1
gradient = np.array([dL_dw1, dL_db1, dL_dw2, dL_db2])

print("순전파 a, 예측값, 손실:", a, y_pred, loss)
print("예측값에 대한 손실 기울기:", dL_dy_pred)
print("w2, b2 기울기:", dL_dw2, dL_db2)
print("a에 전달한 기울기:", dL_da)
print("w1, b1 기울기:", dL_dw1, dL_db1)
print("Gradient [w1, b1, w2, b2]:", gradient)

순전파 a, 예측값, 손실: 5.0 10.0 4.0
예측값에 대한 손실 기울기: -4.0
w2, b2 기울기: -20.0 -4.0
a에 전달한 기울기: -8.0
w1, b1 기울기: -16.0 -8.0
Gradient [w1, b1, w2, b2]: [-16.  -8. -20.  -4.]


#### 풀이 및 답변

**수식 및 계산 과정**

- 순전파: $a=2\times2+1=5$, $\hat y=2\times5+0=10$, $L=(12-10)^2=4$입니다.
- 출력 기울기는 $\partial L/\partial\hat y=2(10-12)=-4$입니다.
- 두 번째 연산: $\partial L/\partial w_2=-4\times5=-20$, $\partial L/\partial b_2=-4$입니다.
- 앞 단계에 전달할 기울기: $\partial L/\partial a=-4\times2=-8$입니다. 따라서 $\partial L/\partial w_1=-8\times2=-16$, $\partial L/\partial b_1=-8$입니다.
- [w₁,b₁,w₂,b₂] 순서의 Gradient는 `[-16, -8, -20, -4]`입니다.

**Q1.** 순전파의 중간값 a, 예측값 ŷ, 손실 L은 각각 얼마인가요?

**답변:** a=5, 예측값 ŷ=10, 손실 L=4입니다.

**Q2.** ∂L/∂ŷ와 두 번째 연산의 ∂L/∂w₂, ∂L/∂b₂는 얼마인가요?

**답변:** 예측값 기울기는 −4입니다. 여기에 a=5를 곱하면 w₂ 기울기는 −20, 1을 곱하면 b₂ 기울기는 −4입니다.

**Q3.** a에 전달한 기울기와 첫 번째 연산의 두 기울기는 얼마인가요?

**답변:** a에 전달하는 기울기는 $-4\times w_2=-8$입니다. w₁ 기울기는 $-8\times x=-16$, b₁ 기울기는 −8입니다.

**Q4.** 역전파를 계산하면 가중치와 편향이 자동으로 바뀌나요?

**답변:** 자동으로 바뀌지 않습니다. 역전파는 파라미터별 기울기를 계산하는 과정입니다. 실제 값을 바꾸려면 학습률과 기울기를 사용하는 업데이트가 별도로 필요합니다.

---

## 필수 실습 2. 이진 분류의 역전파 따라가기

### 문제 2-1. 구매 확률에서 손실 기울기까지 연결하기

#### 문제 설명

고객의 구매 여부를 예측합니다. 실제 정답 y=1은 구매했다는 뜻입니다.

$$z=wx+b,\quad p=\operatorname{Sigmoid}(z)=\frac{1}{1+e^{-z}},\quad
L=-[y\ln p+(1-y)\ln(1-p)]$$

| x | y | w | b |
|---:|---:|---:|---:|
| 2 | 1 | 0.5 | 0 |

#### 요구사항

1. 제공된 코드로 z, 구매 확률 p, BCE 손실을 계산하세요.
2. 아래 미분식에 숫자를 넣어 두 미분값을 곱하세요. 이번 정답은 y=1이므로 ∂L/∂p=−1/p를 사용합니다.
3. w·b의 기울기를 구해 [w,b] 순서로 출력하고 Q1~Q4에 답하세요.

| 계산 대상 | 사용할 식 |
|---|---|
| BCE의 확률에 대한 기울기 | `dL_dp = -1 / p` |
| Sigmoid의 점수에 대한 기울기 | `dp_dz = p * (1 - p)` |
| 점수 z에 대한 손실 기울기 | `dL_dz = dL_dp * dp_dz` |
| w·b에 대한 손실 기울기 | `dL_dw = dL_dz * x`, `dL_db = dL_dz * 1` |

#### 제공 함수와 작성 도움말

```python
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = w * x + b
p = sigmoid(z)
loss = np.logaddexp(0, z) - y * z
```

교안에서 두 미분값의 곱은 p−y로도 정리됩니다. 긴 식을 유도할 필요는 없습니다. 함수와 손실 코드는 그대로 사용하세요. 출력만 소수점 아래 세 자리로 반올림합니다.

#### 해석 질문

**Q1.** z, 구매 확률 p, BCE 손실은 각각 얼마인가요?

**Q2.** ∂L/∂p와 dp/dz는 각각 얼마이고, 왜 곱하나요?

**Q3.** ∂L/∂z와 [w,b] 순서의 Gradient는 얼마인가요?

**Q4.** 현재 두 파라미터 기울기의 음수 부호는 무엇을 의미하나요?

#### 제출 결과

- 요구사항의 계산 결과
- Q1~Q4 답변

In [4]:
# 문제 2-1. 실제 구매 고객이므로 확률에 대한 BCE 미분은 -1/p입니다.
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

x, y = 2.0, 1.0
w, b = 0.5, 0.0
z = w * x + b
p = sigmoid(z)
loss = np.logaddexp(0, z) - y * z

dL_dp = -1 / p
dp_dz = p * (1 - p)
dL_dz = dL_dp * dp_dz
dL_dw = dL_dz * x
dL_db = dL_dz * 1
gradient = np.array([dL_dw, dL_db])

print(f"z={z:.3f}, 구매 확률={p:.3f}, BCE={loss:.3f}")
print(f"dL/dp={dL_dp:.3f}, dp/dz={dp_dz:.3f}")
print(f"dL/dz={dL_dz:.3f}")
print("Gradient [w, b]:", np.round(gradient, 3))

z=1.000, 구매 확률=0.731, BCE=0.313
dL/dp=-1.368, dp/dz=0.197
dL/dz=-0.269
Gradient [w, b]: [-0.538 -0.269]


#### 풀이 및 답변

**수식 및 계산 과정**

- 순전파: $z=0.5\times2+0=1$, $p=1/(1+e^{-1})\approx0.731059$, $L=-\ln p\approx0.313262$입니다.
- $\partial L/\partial p=-1/p\approx-1.367879$, $dp/dz=p(1-p)\approx0.196612$입니다.
- 두 기울기를 곱하면 $\partial L/\partial z\approx-0.268941=p-y$입니다.
- $\partial L/\partial w=(p-y)\times2\approx-0.537883$, $\partial L/\partial b=p-y\approx-0.268941$입니다. 출력의 세 자리 Gradient는 `[-0.538, -0.269]`입니다.

**Q1.** z, 구매 확률 p, BCE 손실은 각각 얼마인가요?

**답변:** z=1.000, 구매 확률 p≈0.731, BCE 손실≈0.313입니다. 손실 계산에는 반올림 전 점수와 확률을 사용합니다.

**Q2.** ∂L/∂p와 dp/dz는 각각 얼마이고, 왜 곱하나요?

**답변:** $\partial L/\partial p\approx-1.368$, $dp/dz\approx0.197$입니다. 점수 z의 변화가 확률 p를 거쳐 손실 L까지 전달되므로 Chain Rule에 따라 두 값을 곱합니다.

**Q3.** ∂L/∂z와 [w,b] 순서의 Gradient는 얼마인가요?

**답변:** 점수 기울기는 약 −0.269이고, [w,b] 순서의 Gradient는 약 `[-0.538, -0.269]`입니다.

**Q4.** 현재 두 파라미터 기울기의 음수 부호는 무엇을 의미하나요?

**답변:** 현재 점에서 다른 파라미터를 고정하고 해당 파라미터를 조금 늘리면 손실이 줄어드는 방향이라는 뜻입니다. 이 예에서는 w와 b를 조금 늘리면 구매 점수와 구매 확률이 높아져 실제 구매 정답에 맞는 쪽으로 변합니다.

### 문제 2-2. 계산 순서표로 ReLU 은닉층까지 따라가기

#### 문제 설명

교안의 작은 신경망입니다. 새로운 미분 규칙을 직접 만들지 않고 아래 순서표를 따라 계산합니다.

$$a=w_1x+b_1,\quad h=\operatorname{ReLU}(a)=\max(0,a)$$
$$z=w_2h+b_2,\quad p=\operatorname{Sigmoid}(z),\quad L=\operatorname{BCE}(y,p)$$

| x | y | w₁ | b₁ | w₂ | b₂ |
|---:|---:|---:|---:|---:|---:|
| 2 | 1 | 1 | 0 | 0.5 | 0 |

h는 이번 문제에서 **ReLU를 거친 중간 출력**입니다. ReLU는 양수를 그대로 내보내고 음수는 0으로 바꿉니다. 현재 a가 양수인지 먼저 확인하세요.

#### 요구사항

1. 순전파로 a, h, z, p, L을 계산하세요. Sigmoid와 BCE 코드는 문제 2-1의 제공 코드를 사용합니다.
2. 아래 표를 한 줄씩 따라 기울기를 구하세요.
3. Gradient를 [w₁,b₁,w₂,b₂] 순서로 출력하고 Q1~Q4에 답하세요. 출력만 세 자리로 반올림합니다.

| 순서 | 계산할 값 | 사용할 코드 |
|---|---|---|
| 1 | 점수에 대한 손실 기울기 | `dL_dz = p - y` |
| 2 | 두 번째 가중치·편향 | `dL_dw2 = dL_dz * h`, `dL_db2 = dL_dz * 1` |
| 3 | h에 대한 손실 기울기 | `dL_dh = dL_dz * w2` |
| 4 | ReLU 미분값 | `relu_grad = 1.0 if a > 0 else 0.0` |
| 5 | a에 대한 손실 기울기 | `dL_da = dL_dh * relu_grad` |
| 6 | 첫 번째 가중치·편향 | `dL_dw1 = dL_da * x`, `dL_db1 = dL_da * 1` |

#### 작성 도움말

순전파의 ReLU는 `h = max(0, a)`로 계산합니다. 표의 조건식은 a가 양수면 1.0, 그렇지 않으면 0.0을 사용한다는 뜻입니다. 교안처럼 a=0에서는 구현상 0을 선택합니다. 이 문제는 a가 양수인 예이며, 0 지점에서 미분이 가능한지 증명할 필요는 없습니다.

#### 해석 질문

**Q1.** 순전파의 a, h, z, p, L은 각각 얼마인가요?

**Q2.** 현재 ReLU 출력과 ReLU 미분값은 각각 얼마인가요?

**Q3.** h와 a에 대한 손실 기울기는 어떻게 구하며 각각 얼마인가요?

**Q4.** [w₁,b₁,w₂,b₂] 순서의 Gradient는 무엇인가요?

#### 제출 결과

- 요구사항의 계산 결과
- Q1~Q4 답변

In [5]:
# 문제 2-2. ReLU 은닉층까지 기울기를 한 단계씩 전달합니다.
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

x, y = 2.0, 1.0
w1, b1, w2, b2 = 1.0, 0.0, 0.5, 0.0
a = w1 * x + b1
h = max(0, a)
z = w2 * h + b2
p = sigmoid(z)
loss = np.logaddexp(0, z) - y * z

dL_dz = p - y
dL_dw2 = dL_dz * h
dL_db2 = dL_dz * 1
dL_dh = dL_dz * w2
relu_grad = 1.0 if a > 0 else 0.0
dL_da = dL_dh * relu_grad
dL_dw1 = dL_da * x
dL_db1 = dL_da * 1
gradient = np.array([dL_dw1, dL_db1, dL_dw2, dL_db2])

print(f"a={a:.3f}, h={h:.3f}, z={z:.3f}, p={p:.3f}, L={loss:.3f}")
print(f"dL/dz={dL_dz:.3f}")
print("w2, b2 기울기:", np.round([dL_dw2, dL_db2], 3))
print(f"dL/dh={dL_dh:.3f}, ReLU 미분={relu_grad:.3f}, dL/da={dL_da:.3f}")
print("w1, b1 기울기:", np.round([dL_dw1, dL_db1], 3))
print("Gradient [w1, b1, w2, b2]:", np.round(gradient, 3))

a=2.000, h=2.000, z=1.000, p=0.731, L=0.313
dL/dz=-0.269
w2, b2 기울기: [-0.538 -0.269]
dL/dh=-0.134, ReLU 미분=1.000, dL/da=-0.134
w1, b1 기울기: [-0.269 -0.134]
Gradient [w1, b1, w2, b2]: [-0.269 -0.134 -0.538 -0.269]


#### 풀이 및 답변

**수식 및 계산 과정**

- 순전파: $a=1\times2+0=2$, $h=\max(0,2)=2$, $z=0.5\times2+0=1$, $p\approx0.731059$, $L\approx0.313262$입니다.
- 출력에서 $\partial L/\partial z=p-y\approx-0.268941$입니다. w₂ 기울기는 약 −0.537883, b₂ 기울기는 약 −0.268941입니다.
- $\partial L/\partial h=(p-y)\times0.5\approx-0.134471$입니다. a>0이므로 ReLU 미분은 1이고, $\partial L/\partial a\approx-0.134471$입니다.
- w₁ 기울기는 $-0.1344707107\times2\approx-0.268941$, b₁ 기울기는 약 −0.134471입니다.
- Gradient의 세 자리 표시는 `[-0.269, -0.134, -0.538, -0.269]`입니다.

**Q1.** 순전파의 a, h, z, p, L은 각각 얼마인가요?

**답변:** a=2.000, h=2.000, z=1.000, p≈0.731, L≈0.313입니다.

**Q2.** 현재 ReLU 출력과 ReLU 미분값은 각각 얼마인가요?

**답변:** ReLU 출력 h는 2이고, 입력 a=2가 양수이므로 ReLU의 미분값은 1입니다. 함수의 출력과 미분값은 구분합니다.

**Q3.** h와 a에 대한 손실 기울기는 어떻게 구하며 각각 얼마인가요?

**답변:** h 기울기는 점수 기울기에 w₂를 곱해 약 −0.134입니다. a 기울기는 여기에 ReLU 미분 1을 곱하므로 역시 약 −0.134입니다.

**Q4.** [w₁,b₁,w₂,b₂] 순서의 Gradient는 무엇인가요?

**답변:** [w₁,b₁,w₂,b₂] 순서로 약 `[-0.269, -0.134, -0.538, -0.269]`입니다. 앞 단계의 w₁·b₁ 기울기에는 출력층 가중치와 ReLU 미분을 거쳐 전달된 값이 반영됩니다.

---

## 과제. 기본 계산 복습

### 문제 3-1. 두 단계 Chain Rule 복습하기

#### 문제 설명

교안 연습문제 1의 함수 $f(x)=(2x+3)^2$에서 x=1을 사용합니다. 문제 1-1과 같은 방법으로 풉니다.

$$z=2x+3,\qquad f=z^2$$

#### 요구사항

1. z와 f를 차례로 계산하세요.
2. df/dz=2z, dz/dx=2를 계산하고 곱해 df/dx를 구하세요.
3. 계산값을 출력하고 Q1~Q4에 답하세요.

#### 작성 도움말

문제 1-1의 코드를 참고하되 안쪽 식을 `z = 2 * x + 3`, 입력을 `x = 1.0`으로 바꾸세요. 안쪽 미분값 `dz_dx`도 이번 식에 맞추어 2로 바꿉니다. 과제에서는 합성함수 하나만 계산하며 추가 회귀 모델이나 세 단계 함수 계산은 요구하지 않습니다.

#### 해석 질문

**Q1.** 중간값 z와 함수값 f는 각각 얼마인가요?

**Q2.** 두 국소 미분값 df/dz와 dz/dx는 각각 얼마인가요?

**Q3.** x=1에서의 전체 미분값 df/dx는 얼마인가요?

**Q4.** 전체 미분값이 양수라는 것은 무엇을 뜻하나요?

#### 제출 결과

- 요구사항의 계산 결과
- Q1~Q4 답변

In [6]:
# 문제 3-1. 안쪽 식의 미분값도 이번 문제에 맞게 바꿉니다.
x = 1.0
z = 2 * x + 3
f = z ** 2
df_dz = 2 * z
dz_dx = 2.0
df_dx = df_dz * dz_dx

print("중간값 z:", z)
print("함수값 f:", f)
print("df/dz:", df_dz)
print("dz/dx:", dz_dx)
print("df/dx:", df_dx)

중간값 z: 5.0
함수값 f: 25.0
df/dz: 10.0
dz/dx: 2.0
df/dx: 20.0


#### 풀이 및 답변

**수식 및 계산 과정**

- $z=2\times1+3=5$, $f=z^2=25$입니다.
- $df/dz=2z=10$, $dz/dx=2$입니다.
- $df/dx=10\times2=20$입니다.

**Q1.** 중간값 z와 함수값 f는 각각 얼마인가요?

**답변:** 중간값 z는 5이고 함수값 f는 25입니다.

**Q2.** 두 국소 미분값 df/dz와 dz/dx는 각각 얼마인가요?

**답변:** $df/dz=10$, $dz/dx=2$입니다.

**Q3.** x=1에서의 전체 미분값 df/dx는 얼마인가요?

**답변:** x=1에서 전체 미분값은 $10\times2=20$입니다.

**Q4.** 전체 미분값이 양수라는 것은 무엇을 뜻하나요?

**답변:** 현재 x=1에서 x를 아주 조금 늘리면 f가 증가하는 방향이라는 뜻입니다. 함수값 자체가 20이라는 뜻은 아닙니다.

### 문제 3-2. 미구매 고객의 기본 Gradient 복습하기

#### 문제 설명

교안 연습문제 3의 미구매 고객입니다. 이번 실제 정답은 y=0입니다.

$$z=wx+b,\qquad p=\operatorname{Sigmoid}(z),\qquad L=\operatorname{BCE}(y,p)$$

| x | y | w | b |
|---:|---:|---:|---:|
| 3 | 0 | 0.2 | 0.1 |

p는 여전히 **구매 확률**이고, 미구매 확률은 1−p입니다. 실제 정답이 0이라고 해서 p의 뜻이 미구매 확률로 바뀌지 않습니다.

#### 요구사항

1. 문제 2-1의 제공 함수로 z, p, 1−p, BCE 손실을 계산하세요.
2. 교안의 정리된 식 `dL_dz = p - y`를 사용하세요.
3. `dL_dw = dL_dz * x`, `dL_db = dL_dz * 1`을 계산해 [w,b] 순서로 출력하세요.
4. Q1~Q4에 답하세요. 출력은 소수점 아래 세 자리로 반올림합니다.

#### 작성 도움말

BCE 코드는 `loss = np.logaddexp(0, z) - y * z`입니다. y=0인 경우 손실의 의미는 −ln(1−p)입니다. 이번 과제는 제공된 p−y 공식을 사용하며 BCE와 Sigmoid의 일반 미분식을 다시 유도하거나 별도로 검증할 필요는 없습니다.

#### 해석 질문

**Q1.** z, 구매 확률 p, 미구매 확률 1−p는 각각 얼마인가요?

**Q2.** BCE는 얼마이며 왜 −ln(1−p)를 사용하나요?

**Q3.** 점수 기울기와 [w,b] 순서의 Gradient는 무엇인가요?

**Q4.** 현재 손실을 줄이려면 각 파라미터를 어느 쪽으로 조금 바꾸나요?

#### 제출 결과

- 요구사항의 계산 결과
- Q1~Q4 답변

In [7]:
# 문제 3-2. y=0이어도 p는 구매 확률을 뜻합니다.
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

x, y = 3.0, 0.0
w, b = 0.2, 0.1
z = w * x + b
p = sigmoid(z)
non_purchase_prob = 1 - p
loss = np.logaddexp(0, z) - y * z
dL_dz = p - y
dL_dw = dL_dz * x
dL_db = dL_dz * 1
gradient = np.array([dL_dw, dL_db])

print(f"z={z:.3f}, 구매 확률={p:.3f}, 미구매 확률={non_purchase_prob:.3f}")
print(f"BCE={loss:.3f}, dL/dz={dL_dz:.3f}")
print("Gradient [w, b]:", np.round(gradient, 3))

z=0.700, 구매 확률=0.668, 미구매 확률=0.332
BCE=1.103, dL/dz=0.668
Gradient [w, b]: [2.005 0.668]


#### 풀이 및 답변

**수식 및 계산 과정**

- $z=0.2\times3+0.1=0.7$, $p=1/(1+e^{-0.7})\approx0.668188$, $1-p\approx0.331812$입니다.
- 실제 정답 y=0이므로 $L=-\ln(1-p)\approx1.103186$입니다.
- $\partial L/\partial z=p-y\approx0.668188$입니다.
- w 기울기는 $0.6681877722\times3\approx2.004563$, b 기울기는 약 0.668188입니다. Gradient의 세 자리 표시는 `[2.005, 0.668]`입니다.

**Q1.** z, 구매 확률 p, 미구매 확률 1−p는 각각 얼마인가요?

**답변:** z=0.700, 구매 확률 p≈0.668, 미구매 확률 1−p≈0.332입니다.

**Q2.** BCE는 얼마이며 왜 −ln(1−p)를 사용하나요?

**답변:** BCE는 약 1.103입니다. 실제 결과가 미구매(0)이므로, 그 결과의 확률인 1−p에 음의 로그를 적용합니다. p는 계속 구매 확률을 뜻합니다.

**Q3.** 점수 기울기와 [w,b] 순서의 Gradient는 무엇인가요?

**답변:** 점수 기울기는 약 0.668이고, [w,b] 순서의 Gradient는 약 `[2.005, 0.668]`입니다.

**Q4.** 현재 손실을 줄이려면 각 파라미터를 어느 쪽으로 조금 바꾸나요?

**답변:** 두 기울기 모두 양수이므로 각각의 파라미터를 조금 줄이는 쪽이 손실을 낮춥니다. 이 예제에서는 w와 b를 줄이면 구매 점수와 구매 확률이 낮아져 실제 미구매 결과에 더 높은 확률을 부여합니다.

---

## 실습 마무리

다음 질문으로 핵심 내용을 복습하세요.

**1.** Chain Rule에서는 연결된 단계의 미분값을 어떻게 사용하나요?

**2.** 순전파와 역전파는 각각 무엇을 구하나요?

**3.** L=(y−ŷ)²의 예측값 기울기에서 −1을 곱하는 이유는 무엇인가요?

**4.** Sigmoid와 BCE를 연결했을 때 p−y는 무엇인가요?

**5.** ReLU 입력 a=2일 때 출력과 미분값은 각각 얼마인가요?

**6.** 역전파와 파라미터 업데이트는 같은 과정인가요?

### 실습 마무리 답변

**1.** **답변:** 연결 경로를 따라 각 단계의 국소 미분값을 곱해 전체 변화율을 구합니다. 역전파에서는 전달받은 손실 기울기에 현재 연산의 미분값을 곱합니다.

**2.** **답변:** 순전파는 입력에서 시작해 중간값·예측·손실을 구합니다. 역전파는 손실에서 거꾸로 이동하며 각 중간값과 파라미터에 대한 손실 기울기를 구합니다.

**3.** **답변:** 안쪽 식 y−ŷ를 ŷ에 대해 미분하면 −1이기 때문입니다. 따라서 제곱의 미분 $2(y-\hat y)$에 −1을 곱해 $2(\hat y-y)$를 얻습니다.

**4.** **답변:** 확률로 바꾸기 전 점수 z에 대한 손실 기울기 $\partial L/\partial z$입니다. 확률 자체나 w의 기울기와는 구분해야 합니다.

**5.** **답변:** 출력은 2이고 미분값은 1입니다. 양수 구간의 ReLU는 입력을 그대로 통과시킵니다.

**6.** **답변:** 서로 다른 과정입니다. 역전파는 기울기를 계산하고, 업데이트는 학습률과 그 기울기를 사용해 파라미터 값을 실제로 변경합니다.